# Agent 2 & Agent 3 Model Ablation Study

Evaluates Agent 2 (Visual Evidence) and Agent 3 (Structured Evidence) across multiple models  
using `claim_explanation_verification_pre_tasksets_test_two_V2.csv` (columns: chart_img, label, claim).

## 1. Setup

In [ ]:
import base64
import csv
import json
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from litellm import completion

load_dotenv(Path("../chart_verifier/.env"))

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY", "")
assert OPENROUTER_KEY, "Set OPENROUTER_API_KEY in chart_verifier/.env"
print("API key loaded.")

## 2. Configuration

In [ ]:
# ── Models to compare (all must support vision via OpenRouter) ────────────────
MODELS = [
    "openrouter/anthropic/claude-sonnet-4-6",
    "openrouter/anthropic/claude-3.7-sonnet",
    "openrouter/google/gemini-2.5-pro",
    "openrouter/openai/gpt-4o",
]

CSV_PATH    = Path("../data/claim_explanation_verification_pre_tasksets_test_two_V2.csv")
IMAGE_CACHE = Path("../eval_image_cache")
IMAGE_CACHE.mkdir(exist_ok=True)

N_CASES    = 30    # number of rows to evaluate (set to None for all)
RATE_DELAY = 2.0   # seconds between API calls
MAX_TOKENS = 1024

print(f"CSV      : {CSV_PATH}")
print(f"Models   : {len(MODELS)}")
print(f"N cases  : {N_CASES}")

## 3. Helper Functions

In [ ]:
def detect_mime(data: bytes) -> str:
    if data[:3] == b"\xff\xd8\xff":        return "image/jpeg"
    if data[:8] == b"\x89PNG\r\n\x1a\n":  return "image/png"
    if data[:6] in (b"GIF87a", b"GIF89a"): return "image/gif"
    if data[:4] == b"RIFF" and data[8:12] == b"WEBP": return "image/webp"
    return "image/jpeg"


def download_image(url: str) -> tuple[str, str] | None:
    fname = url.split("/")[-1].split("?")[0]
    local = IMAGE_CACHE / fname
    if not local.exists():
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "eval/1.0"})
            with urllib.request.urlopen(req, timeout=20) as r:
                local.write_bytes(r.read())
        except Exception as e:
            print(f"  [warn] download failed: {e}")
            return None
    mime = detect_mime(local.read_bytes()[:12])
    return str(local), mime


def encode_b64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def img_part(img_b64: str, mime: str) -> dict:
    return {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{img_b64}"}}


def extract_json_object(text: str) -> dict | None:
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":   depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:    return json.loads(text[start: i + 1])
                except: return None
    return None


def call_model(messages: list, model: str) -> tuple[str, dict]:
    resp = completion(
        model=model,
        messages=messages,
        max_tokens=MAX_TOKENS,
        api_key=OPENROUTER_KEY,
        api_base="https://openrouter.ai/api/v1",
    )
    usage = {
        "prompt_tokens":     getattr(resp.usage, "prompt_tokens", 0),
        "completion_tokens": getattr(resp.usage, "completion_tokens", 0),
    }
    return resp.choices[0].message.content or "", usage


def map_verdict(v: str) -> str:
    """Normalise correct/incorrect → supported/contradicted."""
    v = v.lower()
    if v == "correct":   return "supported"
    if v == "incorrect": return "contradicted"
    return v


def verdict_to_bool(verdict: str) -> bool | None:
    v = verdict.lower()
    if v in ("supported", "correct"):           return True
    if v in ("contradicted", "incorrect"):      return False
    return None  # error / unknown → excluded from metrics


print("Helpers defined.")

## 4. Agent Prompts

In [ ]:
AGENT2_SYSTEM = """You are a Chart Reader Agent. You receive a chart image and a factual claim.

Examine the chart image visually and decide:
  correct   — the chart supports the claim
  incorrect — the chart does not support the claim

"unrelated" is NOT a valid verdict. You MUST choose correct or incorrect.

Confidence scoring rules:
  0.9-1.0 — labels, bars, lines, or colors clearly and directly confirm or deny the claim
  0.6-0.8 — claim is mostly verifiable visually but requires some estimation or interpretation
  0.3-0.5 — chart is ambiguous, crowded, or hard to read for this specific claim
  0.0-0.2 — chart does not contain enough visual information to evaluate this claim

Output ONLY a JSON object — no markdown fences, no extra text:
{{
  "verdict": "<correct | incorrect>",
  "confidence": <float 0.0-1.0>,
  "reasoning": "<1-2 sentences>"
}}

Do not add preamble. Return only the JSON object."""


AGENT3_SYSTEM = """You are a Structured Evidence Agent. You receive a chart image and a factual claim.

Step 1 — Extract a data table:
  Read the chart and extract its data into a structured pseudo-table
  (e.g., rows of "Category | Year | Value"). Estimate from the chart scale if exact
  values are not readable.

Step 2 — Evaluate the claim against the table:
  Decide:
    correct   — the extracted data confirms the claim
    incorrect — the extracted data contradicts the claim

"unrelated" is NOT a valid verdict. You MUST choose correct or incorrect.

Confidence scoring rules:
  0.9-1.0 — exact values read directly from the chart confirm or deny the claim with no ambiguity
  0.6-0.8 — values estimated from scale are close enough to verify the claim with reasonable certainty
  0.3-0.5 — values are difficult to read precisely; significant estimation was needed
  0.0-0.2 — could not extract enough data from the chart to evaluate the claim reliably

Output ONLY a JSON object — no markdown fences, no extra text:
{{
  "extracted_table": "<markdown table>",
  "verdict": "<correct | incorrect>",
  "confidence": <float 0.0-1.0>,
  "reasoning": "<1-2 sentences>"
}}

Do not add preamble. Return only the JSON object."""

print("Prompts defined.")

## 5. Load Dataset

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df_raw)}")
print(f"Columns   : {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
df = df_raw[["chart_img", "label", "claim"]].dropna(subset=["chart_img", "label", "claim"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.upper() == "TRUE"
df = df.reset_index(drop=True)

if N_CASES and len(df) > N_CASES:
    step = len(df) // N_CASES
    df = df.iloc[::step].head(N_CASES).reset_index(drop=True)

print(f"Rows to evaluate : {len(df)}")
print(f"True labels      : {df['label'].sum()} / {len(df)}")
df.head()

## 6. Run Agents for Each Model

In [ ]:
def run_agent(agent_num: int, system_prompt: str, claim: str,
              img_b64: str, mime: str, model: str) -> dict:
    """Call Agent 2 or 3 and return a result dict."""
    time.sleep(RATE_DELAY)
    t0 = time.perf_counter()
    try:
        raw, usage = call_model(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "text", "text": f"Claim: {claim}"},
                    img_part(img_b64, mime),
                ]},
            ],
            model=model,
        )
        obj = extract_json_object(raw) or {}
        verdict = map_verdict(obj.get("verdict", "unknown"))
        if verdict not in ("supported", "contradicted"):
            verdict = "contradicted"
        return {
            "verdict":     verdict,
            "confidence":  float(obj.get("confidence", 0.0)),
            "reasoning":   obj.get("reasoning", raw[:100]),
            "latency_s":   round(time.perf_counter() - t0, 2),
            "tokens":      usage["prompt_tokens"] + usage["completion_tokens"],
            "parse_error": obj == {},
        }
    except Exception as e:
        return {
            "verdict": "error", "confidence": 0.0, "reasoning": str(e),
            "latency_s": round(time.perf_counter() - t0, 2),
            "tokens": 0, "parse_error": True,
        }

In [ ]:
# ── Main evaluation loop ──────────────────────────────────────────────────────
# results shape: { model: { "agent2": [row_dict, ...], "agent3": [row_dict, ...] } }
results = {}

for model in MODELS:
    model_short = model.split("/")[-1]
    print(f"\n{'='*70}")
    print(f"Model: {model_short}")
    print(f"{'='*70}")

    a2_rows, a3_rows = [], []

    for idx, row in df.iterrows():
        claim   = str(row["claim"]).strip()
        img_url = str(row["chart_img"]).strip()
        gt      = bool(row["label"])
        short   = claim[:45] + "..." if len(claim) > 45 else claim

        img_result = download_image(img_url)
        if img_result is None:
            print(f"  [{idx:>3}] SKIP  {short}")
            continue

        img_path, mime = img_result
        img_b64 = encode_b64(img_path)

        # Agent 2
        a2 = run_agent(2, AGENT2_SYSTEM, claim, img_b64, mime, model)
        a2.update({"idx": idx, "claim": claim, "chart_img": img_url, "ground_truth": gt})
        a2_rows.append(a2)

        # Agent 3
        a3 = run_agent(3, AGENT3_SYSTEM, claim, img_b64, mime, model)
        a3.update({"idx": idx, "claim": claim, "chart_img": img_url, "ground_truth": gt})
        a3_rows.append(a3)

        a2_mark = "Y" if verdict_to_bool(a2["verdict"]) == gt else "N"
        a3_mark = "Y" if verdict_to_bool(a3["verdict"]) == gt else "N"
        print(f"  [{idx:>3}] A2:{a2['verdict'][:4]}({a2['confidence']:.2f}){a2_mark}  "
              f"A3:{a3['verdict'][:4]}({a3['confidence']:.2f}){a3_mark}  {short}")

    results[model] = {"agent2": a2_rows, "agent3": a3_rows}

print("\nEvaluation complete.")

## 7. Compute Metrics

In [ ]:
def compute_metrics(rows: list[dict]) -> dict:
    tp = fp = tn = fn = parse_err = 0
    latencies, tokens, confidences = [], [], []

    for r in rows:
        gt   = r["ground_truth"]
        pred = verdict_to_bool(r["verdict"])

        if pred is None or r.get("parse_error"):
            parse_err += 1
            continue

        latencies.append(r["latency_s"])
        tokens.append(r["tokens"])
        confidences.append(r["confidence"])

        if gt and pred:       tp += 1
        elif gt and not pred: fn += 1
        elif not gt and pred: fp += 1
        else:                 tn += 1

    total = tp + fp + tn + fn
    acc   = (tp + tn) / total if total else 0
    prec  = tp / (tp + fp)    if (tp + fp) else 0
    rec   = tp / (tp + fn)    if (tp + fn) else 0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

    true_rows  = [r for r in rows if r["ground_truth"] is True]
    false_rows = [r for r in rows if r["ground_truth"] is False]

    def acc_subset(subset):
        if not subset: return 0.0
        c = sum(1 for r in subset
                if verdict_to_bool(r["verdict"]) == r["ground_truth"]
                and verdict_to_bool(r["verdict"]) is not None)
        return c / len(subset)

    return {
        "accuracy":        acc,
        "precision":       prec,
        "recall":          rec,
        "f1":              f1,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "parse_errors":    parse_err,
        "acc_true":        acc_subset(true_rows),
        "acc_false":       acc_subset(false_rows),
        "avg_latency_s":   round(sum(latencies) / len(latencies), 2) if latencies else 0.0,
        "avg_tokens":      round(sum(tokens) / len(tokens), 0) if tokens else 0.0,
        "avg_confidence":  round(sum(confidences) / len(confidences), 3) if confidences else 0.0,
        "n": total,
    }


# Build summary DataFrame
records = []
for model, data in results.items():
    model_short = model.split("/")[-1]
    for agent_key, label in (("agent2", "Agent 2 (Visual)"), ("agent3", "Agent 3 (Structured)")):
        m = compute_metrics(data[agent_key])
        records.append({"model": model_short, "agent": label, **m})

summary_df = pd.DataFrame(records)
display_cols = ["model", "agent", "accuracy", "precision", "recall", "f1",
                "acc_true", "acc_false", "avg_confidence",
                "avg_latency_s", "avg_tokens", "parse_errors", "n"]
summary_df[display_cols].round(3)

## 8. Visualise Results

In [ ]:
# ── Bar chart: Accuracy / Precision / Recall / F1 per model × agent ──────────
metrics_to_plot = ["accuracy", "precision", "recall", "f1"]
agents          = ["Agent 2 (Visual)", "Agent 3 (Structured)"]
agent_colors    = {"Agent 2 (Visual)": "#4C8BF5", "Agent 3 (Structured)": "#F5A623"}
models          = summary_df["model"].unique().tolist()
n_models        = len(models)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Agent 2 vs Agent 3 — Model Comparison", fontsize=15, fontweight="bold")

for ax, metric in zip(axes.flat, metrics_to_plot):
    x      = np.arange(n_models)
    width  = 0.35
    offset = [-width / 2, width / 2]

    for i, agent in enumerate(agents):
        subset = summary_df[summary_df["agent"] == agent].set_index("model")
        vals   = [subset.loc[m, metric] if m in subset.index else 0 for m in models]
        bars   = ax.bar(x + offset[i], vals, width, label=agent,
                        color=agent_colors[agent], alpha=0.88)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7.5)

    ax.set_title(metric.capitalize(), fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([m[:20] for m in models], rotation=18, ha="right", fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("agent23_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: agent23_metrics.png")

In [ ]:
# ── Accuracy by ground-truth class (True claims vs False claims) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Accuracy Breakdown by Ground-Truth Label", fontsize=13, fontweight="bold")

for ax, metric, title in zip(axes, ["acc_true", "acc_false"],
                               ["Acc on TRUE claims (should predict: supported)",
                                "Acc on FALSE claims (should predict: contradicted)"]):
    x     = np.arange(n_models)
    width = 0.35
    for i, agent in enumerate(agents):
        subset = summary_df[summary_df["agent"] == agent].set_index("model")
        vals   = [subset.loc[m, metric] if m in subset.index else 0 for m in models]
        bars   = ax.bar(x + (i - 0.5) * width, vals, width, label=agent,
                        color=agent_colors[agent], alpha=0.88)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([m[:20] for m in models], rotation=18, ha="right", fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("agent23_acc_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: agent23_acc_breakdown.png")

In [ ]:
# ── Latency & token usage ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cost Proxies: Latency & Token Usage", fontsize=13, fontweight="bold")

for ax, metric, ylabel in zip(axes,
                               ["avg_latency_s", "avg_tokens"],
                               ["Avg Latency (s)", "Avg Tokens"]):
    x     = np.arange(n_models)
    width = 0.35
    for i, agent in enumerate(agents):
        subset = summary_df[summary_df["agent"] == agent].set_index("model")
        vals   = [subset.loc[m, metric] if m in subset.index else 0 for m in models]
        bars   = ax.bar(x + (i - 0.5) * width, vals, width, label=agent,
                        color=agent_colors[agent], alpha=0.88)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([m[:20] for m in models], rotation=18, ha="right", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("agent23_cost.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: agent23_cost.png")

In [ ]:
# ── Confidence distribution per model × agent (box plot) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Confidence Score Distribution", fontsize=13, fontweight="bold")

for ax, agent_key, title in zip(axes,
                                 ["agent2", "agent3"],
                                 ["Agent 2 (Visual)", "Agent 3 (Structured)"]):
    data_per_model = []
    labels         = []
    for model in models:
        rows = results[model][agent_key]
        confs = [r["confidence"] for r in rows if not r.get("parse_error")]
        data_per_model.append(confs)
        labels.append(model.split("/")[-1][:18])

    bp = ax.boxplot(data_per_model, labels=labels, patch_artist=True,
                    boxprops=dict(facecolor=agent_colors[title], alpha=0.7))
    ax.set_title(title, fontsize=11)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Confidence")
    ax.tick_params(axis="x", rotation=18, labelsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("agent23_confidence_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: agent23_confidence_dist.png")

In [ ]:
# ── F1 heatmap ────────────────────────────────────────────────────────────────
pivot = summary_df.pivot(index="agent", columns="model", values="f1")
pivot.columns = [c[:20] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=18, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title("F1 Score Heatmap (Agent × Model)", fontsize=12, fontweight="bold")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=10, color="black" if 0.3 < val < 0.85 else "white")

plt.tight_layout()
plt.savefig("agent23_f1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: agent23_f1_heatmap.png")

## 9. Final Summary Table

In [ ]:
print("\n" + "="*110)
print("AGENT 2 & AGENT 3 — MODEL COMPARISON SUMMARY")
print("="*110)
header = (f"{'Model':<24} {'Agent':<22} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}  "
          f"{'AccT':>6} {'AccF':>6}  {'Conf':>6} {'Err':>4}  {'Lat(s)':>7} {'Tokens':>7}")
print(header)
print("-"*110)

for _, row in summary_df.iterrows():
    print(
        f"{row['model']:<24} {row['agent']:<22} "
        f"{row['accuracy']:>6.3f} {row['precision']:>6.3f} {row['recall']:>6.3f} {row['f1']:>6.3f}  "
        f"{row['acc_true']:>6.3f} {row['acc_false']:>6.3f}  "
        f"{row['avg_confidence']:>6.3f} {int(row['parse_errors']):>4}  "
        f"{row['avg_latency_s']:>7.1f} {int(row['avg_tokens']):>7}"
    )

print("="*110)
print("AccT = accuracy on TRUE claims | AccF = accuracy on FALSE claims | Conf = avg confidence")

# Best model per agent
for agent in agents:
    sub  = summary_df[summary_df["agent"] == agent]
    best = sub.loc[sub["f1"].idxmax()]
    print(f"\nBest {agent}: {best['model']}  (F1 = {best['f1']:.3f})")

## 10. Save Detailed Results

In [ ]:
detail_rows = []
for model, data in results.items():
    for agent_key in ("agent2", "agent3"):
        for r in data[agent_key]:
            detail_rows.append({"model": model, "agent": agent_key, **r})

detail_df = pd.DataFrame(detail_rows)
out_path  = "eval_agent23_results.csv"
detail_df.to_csv(out_path, index=False)
print(f"Detailed results saved to {out_path}  ({len(detail_df)} rows)")
detail_df.head()